# 7. What happens when two plane waves overlap?

[Notebook 5](../5_Grating_Diffraction_in_Three_Dimensions.ipynb) established the allowed directions of diffraction orders and which an optical system can collect. [Notebook 6](../6_Amplitude_and_Intensity_of_a_Scalar_Wave.ipynb) examined the intensity recorded from one selected order: its amplitude sets its brightness, while changing its constant phase offset leaves that brightness unchanged.

This leaves a new question:

> **What happens when two diffraction orders overlap?**

Before returning to the grating, consider a related experiment that prepares two beams with a beam splitter. It lets us begin with the more general question:

> **What happens when two plane waves overlap at a detector?**

**Scope of the main experiment:** Sections 1–3 demonstrate fully coherent two-beam interference, assuming perfect temporal coherence and deterministic spatial phase relationships. They do **not** measure the source's temporal self-coherence versus optical path delay. Sections 4 and 5 separately explore imposed phase averaging and spatial coherence.

Run the code cells in order. Explore the apparatus first, then view the intensity on its detector. Mirror-tilt and detector-position controls sit beside the detector view; the apparatus drawing follows the same settings. No earlier notebook needs to be executed.

In [ ]:
from pathlib import Path
import sys
import importlib
import matplotlib.pyplot as plt
from IPython.display import display

_support_dir = next((base
    for base in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (base / 'wave_optics' / '__init__.py').is_file()), None)
if _support_dir is None:
    raise FileNotFoundError('Open this notebook within the Grating_Physics folder.')
if str(_support_dir) not in sys.path:
    sys.path.insert(0, str(_support_dir))
importlib.invalidate_caches()
if 'wave_optics' in sys.modules:
    sys.modules['wave_optics'].__path__[:] = [str(_support_dir / 'wave_optics')]

# Release explorers if this kernel previously ran the earlier notebook draft.
if 'two_wave_lessons' in globals():
    for old_lesson in two_wave_lessons.values():
        old_lesson.close()
    two_wave_lessons = {}

if 'mirror_tilt_explorer' in globals():
    mirror_tilt_explorer.close()

if 'point_interference_explorer' in globals():
    point_interference_explorer.close()

import wave_optics.interference_setup as interference_setup
importlib.reload(interference_setup)

import wave_optics.detector_interference as detector_interference
importlib.reload(detector_interference)

import wave_optics.phase_geometry as phase_geometry
importlib.reload(phase_geometry)
import wave_optics.interference_wavefronts as interference_wavefronts
importlib.reload(interference_wavefronts)

import wave_optics.phase_exposure as phase_exposure
importlib.reload(phase_exposure)

if 'spatial_coherence_explorer' in globals():
    spatial_coherence_explorer.close()

import wave_optics.spatial_coherence as spatial_coherence
importlib.reload(spatial_coherence)


## 1. A related experiment: split one beam into two

Send a collimated laser beam onto a beam splitter. The transmitted beam travels along $-z$ to the detector. The reflected part travels along $+x$ to a mirror, which redirects it toward the same detector region.

Set the splitter at $S=(0,0,0)$, the mirror at $M=(h,0,0)$ and the detector centre at $P=(0,0,z_D)$, with $z_D<0$. The detector is an $xy$ plane. In the side view below, $x$ points right, $z$ points up, and $+y$ points into the page.

Over a sufficiently small overlap region, approximate each beam by a plane wave. The Matplotlib sketch shows beam axes and finite beam-width contours.

**Give the laser physical specifications**

Use a continuous-wave (CW) laser, which emits continuously during the observation. A concrete starting example is:

| Source parameter | Example value | What it tells us |
| --- | --- | --- |
| Wavelength $\lambda$ | $632.8\,\mathrm{nm}$ | The spatial scale of the optical wave; it will help set the fringe spacing for a given crossing angle. |
| Output power $P$ | $1.0\,\mathrm{mW}$ | The total optical power available before splitting. For a fixed beam profile, field amplitude scales as $\sqrt{P}$, connecting to notebook 6. |
| Output beam diameter $D$ | $0.54\,\mathrm{mm}$, at $1/e^2$ intensity | The characteristic transverse size of the beam at the laser output. Finite beam widths determine the area over which the two beams can overlap. |

These example values follow the [Newport N-STP-912 He-Ne laser specifications](https://www.newport.com/f/stabilized-hene-lasers). For a Gaussian beam, the $1/e^2$ diameter is measured between points where intensity falls to about $13.5\%$ of its central value; it is not a sharp beam edge. The quoted output diameter need not equal the diameter at the detector.

Assume approximately collimated beams with **linear polarization along $y$, perpendicular to the $xz$ sketch**. Their common polarization direction factors out, so we can describe each beam by a scalar field. The plane-wave approximation applies locally within their overlap.

A split unpolarized beam can also show interference. When its two orthogonal polarization components produce equivalent interference patterns, a scalar model can represent their summed intensity. For further explanation, see TU Delft's [Interference and polarisation](https://interactivetextbooks.tudelft.nl/tn2421/content/Chap5_Interference/InterferenceCoherence_2022_01Clean.html#interference-and-polarisation), including the Fresnel-Arago laws.

The three values below specify the example source. The apparatus drawing shows beam-width contours; the detector calculation below assigns a length scale and uses a fixed beam profile. Changing the wavelength represents choosing a different source, rather than continuously tuning this He-Ne laser.

**Mirror tilt and crossing angle**

$\theta$ is the full angle between the beams. Tilting the mirror by $\delta$ rotates beam 2 by $2\delta$. Positive tilt (counterclockwise in the $xz$ side view) decreases the crossing angle:

$$
\theta = \theta_0 - 2\delta,
$$

where $\theta_0$ is the angle at zero tilt.


**Detector position along $z$**

The detector lies at $z=z_D<0$, measured from the splitter in schematic units. Making $z_D$ **more negative** moves it farther along the beams. Its centre remains at $x=y=0$, and its plane stays parallel to $xy$.


**Overlap volume and detector slice**

Approximate each beam near the crossing by a circular cylinder of constant diameter $D_c$. For equal diameters and axes crossing at the full angle $\theta$, their common volume is

$$
V_{\mathrm{overlap}}=\frac{2D_c^3}{3|\sin\theta|}.
$$

The cylinders must extend through the whole intersection. This geometric result is given by [Hubbell (1965)](https://nvlpubs.nist.gov/nistpubs/jres/69C/jresv69Cn2p139_A1b.pdf).

Here we **assume $D_c=0.54\,\mathrm{mm}$ at the crossing**, separately from the source specification. At zero tilt, $\theta\approx16.93^\circ$ and $V_{\mathrm{overlap}}\approx0.361\,\mathrm{mm}^3$. For Gaussian beams, the chosen $1/e^2$ contour defines this volume; tails extend outside it.

**Try:** tilt the mirror and watch the volume estimate change. Then hold the mirror fixed and move the detector: the volume stays unchanged, while the detector samples a different two-dimensional slice.

In [ ]:
# Example source specifications at the laser output.
laser_wavelength_nm = 632.8
laser_power_mw = 1.0
laser_diameter_mm = 0.54  # 1/e^2 intensity diameter

# Separate assumption: equal, constant beam diameters near the crossing.
crossing_diameter_mm = 0.54  # mm; not calculated from the output diameter

# Explicit assumptions for the detector calculation below.
mm_per_drawing_unit = 1.8  # 0.30 drawing-unit beam diameter becomes 0.54 mm
optics_phase_deg = 0.0    # assumed combined reflection/transmission phase offset

# Release the previous display when rerunning this cell.
if 'mirror_tilt_explorer' in globals():
    mirror_tilt_explorer.close()
if 'plane_wave_setup_figure' in globals():
    plt.close(plane_wave_setup_figure)

mirror_tilt_explorer = interference_setup.MirrorTiltExplorer(
    wavelength_nm=laser_wavelength_nm,
    power_mw=laser_power_mw,
    diameter_mm=laser_diameter_mm,
    crossing_diameter_mm=crossing_diameter_mm,
    mm_per_unit=mm_per_drawing_unit,
    optics_phase_deg=optics_phase_deg,
)
display(mirror_tilt_explorer.ui)

*Source:* This simplified arrangement is inspired by T. Catunda, J. Sartori and L. A. O. Nunes, [“Plane wave interference: A didactic experiment to measure the wavelength of light”](https://www.researchgate.net/publication/243492194_Plane_wave_interference_A_didactic_experiment_to_measure_the_wavelength_of_light), *American Journal of Physics* **66**, 548–549 (1998), DOI: 10.1119/1.18825. The original experiment also uses a reticle and a lens to measure and magnify the pattern; this drawing focuses on preparing and overlapping the two beams.

## 2. Wavefronts and the detector

Recall Notebook 5's wavefront planes. Here blue and orange crest sheets approach the **same detector patch**, shown in 3D on the left and in $xy$ on the right. Select either signed field or **Combined intensity**; the lower-right profile always shows intensity through $P$.

The patch is $10\lambda$ square. In the 3D view, $z-z_D=0$ is the detector and $z-z_D>0$ is the incoming side. Each field snapshot puts beam 1 at a crest at $P$, preserving the apparatus's relative phase. Drag to rotate; open **Full detector and enlarged patch** to see the beam footprints.

The vector inset places $\mathbf k_1$ and $\mathbf k_2$ at a common origin: the arrow **from tip 1 to tip 2** is $\Delta\mathbf k$. Vectors are scaled by $k$. The **mirror tilt** and **detector position** controls are here, beside the measured pattern; they also update the apparatus drawing.

**Try:** set $\delta=+0.5^\circ$ to decrease the crossing angle by $1^\circ$. Then set $\delta=+2^\circ$ and move the detector to $z_D\approx-9.15$ to restore centred overlap.


In [ ]:
# Wavefronts, matching detector slice, wavevector difference and intensity profile.
display(mirror_tilt_explorer.detector_ui)


## 3. From wavefront geometry to intensity

**Read phase from the wavefront planes**

Each beam has phase $\Phi_j(\mathbf r,t)=\mathbf k_j\cdot\mathbf r-\omega t+\phi_{j0}$, constant on planes perpendicular to $\mathbf k_j$. Because the beams share the same frequency,

$$
\boxed{\Delta\Phi(\mathbf r)
=(\mathbf k_2-\mathbf k_1)\cdot\mathbf r+\Delta\phi_0,}
\qquad \Delta\phi_0=\phi_{20}-\phi_{10}.
$$

Thus $\Delta\mathbf k=\mathbf k_2-\mathbf k_1$ is perpendicular to the planes of constant **relative phase**. Where the beams overlap, the detector cuts these planes into fringes.

**Use our two beam directions**

With $k=2\pi/\lambda$, beam 1 travels along $-z$ and beam 2 approaches from the mirror at positive $x$:

$$
\begin{aligned}
\mathbf k_1=(k_{1x},k_{1y},k_{1z})&=k(0,0,-1),\\
\mathbf k_2=(k_{2x},k_{2y},k_{2z})&=k(-\sin\theta,0,-\cos\theta).
\end{aligned}
$$

These describe propagation **after the splitter and mirror**. Both beams have $k_y=0$, and their negative $k_z$ components specify propagation toward decreasing $z$.

The tip-to-tip construction in the inset gives

$$
\Delta\mathbf k=\mathbf k_2-\mathbf k_1
=k(-\sin\theta,0,1-\cos\theta).
$$

**Sample the detector plane**

At $\mathbf r=(x,y,z_D)$, with $x=y=0$ at detector centre $P$,

$$
\begin{aligned}
\Delta\Phi(x,y,z_D)
&=(k_{2x}-k_{1x})x+(k_{2z}-k_{1z})z_D+\Delta\phi_0\\
&=k\left[-x\sin\theta+z_D(1-\cos\theta)\right]+\Delta\phi_0.
\end{aligned}
$$

There is no phase variation along $y$. For our apparatus, with splitter-to-mirror separation $h$ along $+x$,

$$
\Delta\phi_0=kh(1+\sin\theta)+\Delta\phi_{\mathrm{optics}},
$$

where $\Delta\phi_{\mathrm{optics}}$ is the relative phase shift from the splitter and mirror. The offset is constant across space for a fixed mirror setting; its derivation is optional below.

Moving the detector changes $z_D$; moving across the fringes changes $x$. Tilting the mirror changes $\theta$ and the apparatus phase offset.

At fixed tilt, moving the detector changes overlap and fringe position, while $\Delta\mathbf k$ stays fixed. The fringe spacing across $x$ therefore stays fixed:

$$
\Lambda_x=\frac{2\pi}{|\Delta k_x|}=\frac{2\pi}{|k_{2x}-k_{1x}|}=\frac{\lambda}{|\sin\theta|}.
$$

At the default angle, $\Lambda_x\approx2.17\,\mu\mathrm m$: about 250 periods fit across the beam diameter. The full view averages over image bins, so the fine stripes are resolved in the close-up. The fringes run along $y$, perpendicular to their variation across $x$.


In [ ]:
# Optional: trace the arms to derive the phase offset.
display(mirror_tilt_explorer.phase_geometry_ui)

**From phase to intensity**

At one detector point $(x,y,z_D)$, let $U_1(t)$ and $U_2(t)$ be the arriving fields of our two arms, with the common polarization assumed above. We now allow their amplitudes and phases to fluctuate. In Notebook 6's scalar-intensity normalization, the detector averages the squared field sum:

$$
\begin{aligned}
\bar I
&=\left\langle |U_1+U_2|^2\right\rangle\\
&=\left\langle (U_1+U_2)(U_1^*+U_2^*)\right\rangle\\
&=\left\langle |U_1|^2\right\rangle+\left\langle |U_2|^2\right\rangle
+\left\langle U_1^*U_2\right\rangle+\left\langle U_1U_2^*\right\rangle\\
&=\bar I_1+\bar I_2+2\operatorname{Re}\left\langle U_1^*U_2\right\rangle,
\qquad \bar I_j=\left\langle |U_j|^2\right\rangle.
\end{aligned}
$$

The star denotes complex conjugation; brackets denote averaging over the observation. The last term is what survives from interference. It depends on **how the two fields fluctuate together**, as well as on their individual intensities.

**Why introduce a degree of coherence?**

Interference experiments show a continuous range of fringe contrast as path difference or source extent changes. Following van Cittert's 1934 work on spatial coherence, Zernike (1938) connected a quantitative degree of coherence to fringe visibility. Wolf's work in the 1950s developed the description in terms of observable field correlations. See [Gbur and Visser, §§2–3](https://www.nat.vu.nl/~tvisser/PinO.pdf). [Thompson and Wolf (1957)](https://opg.optica.org/josa/abstract.cfm?uri=josa-47-10-895) then compared the partial-coherence interference law with measured changes in fringe visibility.

Our calculation exposes the same experimental question: **how much interference remains after averaging?**


**The correlation already present in the intensity**

Name the averaged cross term

$$
\Gamma_{12}=\left\langle U_1^*(t)U_2(t)\right\rangle.
$$

This is the mutual coherence of the two arriving fields at zero extra time delay. Its magnitude also depends on brightness: simply attenuating either arm reduces $|\Gamma_{12}|$. To measure correlation independently of that intensity scale, normalize it:

$$
\boxed{\gamma_{12}=\frac{\Gamma_{12}}{\sqrt{\bar I_1\bar I_2}}},
\qquad
|\Gamma_{12}|^2\leq\bar I_1\bar I_2
\quad\Longrightarrow\quad
0\leq|\gamma_{12}|\leq1.
$$

The inequality is Cauchy–Schwarz; the normalization assumes both arm intensities are nonzero. This **complex degree of mutual coherence** includes amplitude and phase fluctuations. Its magnitude is unchanged by fixed attenuation of either arm.

| Magnitude | Relation between the two fields |
| --- | --- |
| $\lvert\gamma_{12}\rvert=1$ | Full mutual coherence: perfect field correlation |
| $0<\lvert\gamma_{12}\rvert<1$ | Partial mutual coherence |
| $\lvert\gamma_{12}\rvert=0$ | No first-order mutual coherence; the averaged cross term vanishes |

Full mutual coherence means $U_2(t)=cU_1(t)$ with a fixed complex factor $c$ over the averaging interval.

Writing $\gamma_{12}=|\gamma_{12}|e^{i\alpha_{12}}$ gives the general two-beam law:

$$
\boxed{
\bar I=\bar I_1+\bar I_2+
2\sqrt{\bar I_1\bar I_2}\,|\gamma_{12}|\cos\alpha_{12},
\qquad \alpha_{12}=\arg\gamma_{12}.
}
$$

Here the fields are evaluated **at the detector**, so $\alpha_{12}$ already includes their propagation and optical phase shifts.

**Recover our steady-beam model**

For the displayed beams, $U_j=\sqrt{I_j}e^{i\Phi_j}$ with steady amplitudes and relative phase. Hence

$$
\gamma_{12}
=\frac{U_1^*U_2}{\sqrt{I_1I_2}}
=e^{i(\Phi_2-\Phi_1)}
=e^{i\Delta\Phi(x,y,z_D)},
$$

so $|\gamma_{12}|=1$, $\bar I_j=I_j$, and

$$
I(x,y,z_D)=I_1+I_2+2\sqrt{I_1I_2}\cos\Delta\Phi(x,y,z_D).
$$

Our apparatus supplies $\Delta\Phi$: its variation across $x$ places the bright and dark fringes. Its stability during the observation preserves the interference term. The unit coherence magnitude follows from the prescribed fields; it is a model assumption, not a value inferred from a delay scan.

<details>
<summary>Comparing fields at a time delay</summary>

For stationary fields, the general correlation is

$$
\Gamma_{12}(\tau)=\left\langle U_1^*(t)U_2(t+\tau)\right\rangle,
\qquad
\gamma_{12}(\tau)=\frac{\Gamma_{12}(\tau)}{\sqrt{\bar I_1\bar I_2}}.
$$

Here $\tau$ is an additional comparison delay. Our detector intensity uses $\tau=0$ because $U_1$ and $U_2$ are the fields arriving simultaneously; their travel through the arms is already included. A sufficiently long time average agrees with the ensemble average when the fields are ergodic.

</details>


**What does the detector tell us about coherence?**

Define fringe visibility by comparing a maximum and a minimum. During a phase scan with fixed local arm intensities and fixed $|\gamma_{12}|$,

$$
\bar I_{\max,\min}
=\bar I_1+\bar I_2
\ \pm\ 2\sqrt{\bar I_1\bar I_2}\,|\gamma_{12}|.
$$

Therefore,

$$
\boxed{
V=\frac{\bar I_{\max}-\bar I_{\min}}{\bar I_{\max}+\bar I_{\min}}
=\underbrace{\frac{2\sqrt{\bar I_1\bar I_2}}{\bar I_1+\bar I_2}}_{\text{intensity balance}}
\underbrace{|\gamma_{12}|}_{\text{mutual coherence}}.
}
$$

For equal intensities, $V=|\gamma_{12}|$: visibility directly measures the coherence magnitude. Unequal intensities reduce contrast even for fully coherent beams; for example, $\bar I_1=4\bar I_2$ and $|\gamma_{12}|=1$ give $V=0.8$. The same formula applies locally to spatial fringes when the beam envelopes and coherence magnitude change little over one fringe. See [Agarwal and Classen, §I, Eq. (4)](https://arxiv.org/pdf/2001.05561).

**In the steady-beam plots above, $|\gamma_{12}|=1$ throughout.** Moving the detector can weaken beam 2 relative to beam 1 near $P$, reducing contrast through intensity imbalance. The full-detector image can also average over unresolved fringes. Neither effect changes the field coherence.

**An interferometer geometry is not, by itself, a coherence measurement.** The arms carry delayed copies of the same source field, so the apparatus could investigate temporal self-coherence. In this simulation, however, the fields are prescribed as monochromatic with deterministic spatial phase relationships. Path changes alter their calculated phase; perfect temporal coherence is assumed:

$$
|\gamma_{\mathrm{self}}(\tau)|=1
\quad\text{for every delay }\tau.
$$

The source's coherence magnitude is fixed by assumption rather than inferred from a path-delay scan. These plots therefore **demonstrate coherent interference; they do not measure a temporal self-coherence function or determine a coherence time**.

A path-delay change alone preserves visibility when overlap, intensity balance, polarization and resolution are maintained. Our tilt and detector-position controls also change geometry. Studying a finite coherence time would require a source model with finite bandwidth and a delay-dependent correlation calculation; see [TU Delft: temporal coherence](https://interactivetextbooks.tudelft.nl/tn2421/content/Chap5_Interference/InterferenceCoherence_2022_01Clean.html#temporal-coherence-and-the-michelson-interferometer).


**Parameters used in the plots**

For this calculation, choose **1 drawing unit $=1.8\,\mathrm{mm}$** and **$\Delta\phi_{\mathrm{optics}}=0$**, with ideal 50:50 splitting and a lossless mirror. Use constant Gaussian beam profiles with the assumed $D_c=0.54\,\mathrm{mm}$; their smooth tails extend beyond the cylindrical contours. The detector is a $2.34\,\mathrm{mm}$ square.

<details>
<summary>Beam profile and intensity scale</summary>

For either arm, $I_j(\rho_j)=I_{\mathrm{peak}}\exp(-2\rho_j^2/w^2)$, where $w=D_c/2$ and $\rho_j$ is distance perpendicular to that beam's axis. Width and planar phase fronts are held fixed; beam spreading and wavefront curvature are not calculated.

One intensity unit is the on-axis scalar intensity in either arm for the default 1 mW source, ideal 50:50 splitting and $D_c=0.54\,\mathrm{mm}$. This reference and the display range remain fixed while moving the two sliders. These are ideal scalar intensities, not calibrated camera counts.

</details>

**Further reading:** [Gbur and Visser (2010), §§2–3](https://www.nat.vu.nl/~tvisser/PinO.pdf) connect field correlations and Young's experiment, then develop the frequency-domain description. [Agarwal and Classen (2020), §I](https://arxiv.org/pdf/2001.05561) review the normalized coherence function, fringe visibility and Wolf's influence on modern optics. Goodman, *Statistical Optics*, 2nd ed., §§5.2.2–5.2.4, gives a fuller derivation.


**Try:** at zero tilt, move $z_D$ from $-6.9$ towards $-5.5$: the footprints separate and fringe contrast near $P$ falls. Set the tilt to $+2^\circ$, then move $z_D$ to about $-9.15$ to bring the axes together again.

**Next question:** what happens to the recorded fringes if the relative phase changes during the detector's exposure?


## 4. Control the EO phase; observe the detector

Place an ideal **electro-optic (EO) phase modulator** in beam 2, between the splitter and mirror. Voltage changes the crystal's refractive index and optical path length, adding phase. This model holds the beam direction and intensity fixed. See [how an EO phase modulator works](https://media.thorlabs.com/contentassets/5924f547c0be4d98a41aee67e7a83d0c/fiber_eo_phase_modulator_lab_fact.pdf).

The drawing shows the same apparatus with the modulator added. Its controls are beside the detector view that follows.


In [ ]:
# Apparatus drawing with the EO phase modulator.
display(mirror_tilt_explorer.modulated_ui)


Use **mirror tilt**, **detector position** and **EO phase** beside the detector below. The EO slider adds $\beta$ to the orange wave: its crest planes and detector fringes shift together, while $\Delta\mathbf k$ stays fixed.

The view below holds the EO phase steady. Open **Optional: phase changes during exposure** beneath it to compare a switching fringe pattern with its exposure average.


In [ ]:
# Matching wavefront and detector views, with controls at the measurement.
display(mirror_tilt_explorer.detector_modulated_ui)


**First, hold the phase steady.** Move the EO slider from $0^\circ$ to $180^\circ$. The fringes shift, but their contrast stays unchanged:

$$
U_2\longrightarrow U_2e^{i\beta},
\qquad
\gamma_{12}=e^{i(\Delta\Phi+\beta)},
\qquad
|\gamma_{12}|=1.
$$

**Then, open the optional exposure example, set EO phase to $180^\circ$ and press Play.** On the left, the orange wavefronts and current fringes switch between the two phase settings. On the right, the detector records their equal-time average: bright and dark stripes exchange places, so the average has no fringes.

Pause to inspect either setting, or select it directly. Playback is slowed for illustration; the right panel always shows the completed exposure. Try $90^\circ$: some fringe contrast survives.

**What changes here?** EO switching can reduce the correlation between the two arms averaged over the exposure. It does not measure the source's intrinsic coherence time. Switching phase introduces additional frequency components, so the modulated field is no longer strictly monochromatic over the full switching cycle; see [Thorlabs, “Modulation Depth and Frequency Sidebands”](https://media.thorlabs.com/contentassets/5924f547c0be4d98a41aee67e7a83d0c/fiber_eo_phase_modulator_lab_fact.pdf#page=3).

<details>
<summary>Connect this averaging to the degree of coherence</summary>

The two phase settings have equal weights, so

$$
\gamma_{12}
=e^{i\Delta\Phi}\frac{1+e^{i\beta}}{2},
\qquad
\boxed{|\gamma_{12}|=\left|\cos\frac{\beta}{2}\right|}.
$$

For equal local arm intensities, $V=|\gamma_{12}|$; otherwise the intensity-balance factor still applies. A short exposure containing only one phase setting would preserve the steady fringe contrast.

The apparatus phase $\Delta\Phi$ includes any fixed delay through the modulator; $\beta$ is the additional controlled phase.

</details>


## 5. Spatial coherence: sample the field at two openings

The earlier experiment compared two arms at one detector point. **Spatial coherence** asks how strongly the field at two different positions is correlated. A spatial fringe pattern alone does not tell us which kind of coherence we are testing.

Use a related Young experiment: a **uniform, narrowband, spatially incoherent line source** of width $a$ illuminates two small openings separated by $d$. Its source points fluctuate independently; each point illuminates both openings and produces its own shifted fringe pattern. The detector adds the intensities from these independent contributions.

Light still travels towards $-z$: the source is at $z=R$, the openings are $Q_1=(-d/2,0,0)$ and $Q_2=(d/2,0,0)$, and the detector is at $z=-L$. Assume common polarization, small angles, equal arm intensities and sufficient temporal coherence for the path differences. We show the central detector region, where the small-opening diffraction envelope is nearly constant.


In [ ]:
# A separate Young experiment, using the same example wavelength.
if 'spatial_coherence_explorer' in globals():
    spatial_coherence_explorer.close()

spatial_coherence_explorer = spatial_coherence.SpatialCoherenceExplorer(
    wavelength_nm=laser_wavelength_nm,
    source_distance_mm=500.,
    detector_distance_mm=1000.,
)
display(spatial_coherence_explorer.ui)


**Try:**

- Start with $a=0$ (the point-source limit). Change $d$: the fringe spacing changes, but visibility stays at one.
- Set $d=0.500\,\mathrm{mm}$ and increase $a$ to about $0.633\,\mathrm{mm}$. The shifted patterns nearly cancel: contrast falls to zero while fringe spacing stays fixed.
- Increase $a$ further. Weak fringes return, shifted by half a period. The visibility curve shows these revivals.

Source brightness is normalized as its width changes. Increasing the width of a **coherent laser beam** would not produce this loss of spatial coherence; the source points here are mutually incoherent.


**From spatial correlation to the recorded fringes**

Apply the same correlation definition to the field **at the openings**, before propagation to the detector:

$$
\gamma_{Q_1Q_2}
=\frac{\left\langle U^*(Q_1,t)U(Q_2,t)\right\rangle}
{\sqrt{I(Q_1)I(Q_2)}}.
$$

For this centred, uniform line source,

$$
q=\frac{ad}{\lambda R},
\qquad
\gamma_{Q_1Q_2}=\frac{\sin(\pi q)}{\pi q},
\qquad \gamma_{Q_1Q_2}=1\ \text{at }q=0.
$$

Propagation adds the relative phase $-kdx/L$, where $k=2\pi/\lambda$. With equal local arm intensities,

$$
\frac{I(x)}{I_1+I_2}
=1+\operatorname{Re}\!\left\{\gamma_{Q_1Q_2}e^{-ikdx/L}\right\},
\qquad
V=|\gamma_{Q_1Q_2}|,
\qquad
\Lambda_x=\frac{\lambda L}{d}.
$$

Thus source width controls the correlation, while opening separation controls both correlation and fringe spacing. Here $\gamma_{Q_1Q_2}$ is real; a negative value shifts the fringes by half a period. The first zero occurs at $ad=\lambda R$.


<details>
<summary>Where does the source-width formula come from?</summary>

A source point at transverse position $\xi$ gives the openings a relative phase $-kd\xi/R$. Independent, equally bright source points contribute additively to the mutual correlation, giving

$$
\gamma_{Q_1Q_2}
=\frac{1}{a}\int_{-a/2}^{a/2}
e^{-ikd\xi/R}\,d\xi
=\frac{\sin\!\left(\pi ad/(\lambda R)\right)}
{\pi ad/(\lambda R)}.
$$

The detector profile averages this **continuous** source. The orange curves show only a few example source points.

</details>

For further explanation, see TU Delft's [“Spatial Coherence and Young's Experiment”](https://interactivetextbooks.tudelft.nl/tn2421/content/Chap5_Interference/InterferenceCoherence_2022_01Clean.html), and Sections 2–3 of Gbur and Visser's [*The Structure of Partially Coherent Fields*](https://www.nat.vu.nl/~tvisser/PinO.pdf).


## 6. What the cases so far tell us

**Lower fringe visibility does not, by itself, establish lower coherence.** All cases here assume common polarization.

| Case | What happens to visibility? | What it tells us about coherence |
| --- | --- | --- |
| Steady monochromatic beams, equal local intensities, resolved fringes | $V=1$. | Full mutual coherence between the arriving beams is assumed. |
| Tilt or detector motion makes the local arm intensities unequal | $V$ can fall. | The steady fields remain fully mutually coherent. |
| The full-detector image averages over fine fringes | Recorded contrast can fall. | Field coherence is unchanged; the measurement averages bright and dark regions. |
| EO phase is changed, then held steady | Fringes shift; visibility is unchanged at fixed geometry. | The phase of $\gamma_{12}$ changes; its magnitude remains one. |
| EO phase switches during the exposure | Averaged contrast can fall to zero. | The magnitude of the exposure-averaged mutual correlation can decrease. |
| Source width or opening separation changes in the Young experiment | Visibility follows $\lvert\gamma_{Q_1Q_2}\rvert$, including zeros and revivals. | This probes spatial coherence between the two sampled positions. |

**Keep the labels connected:** mutual coherence describes the correlation between the two field samples; spatial coherence specifies that we sample different positions. Self-coherence compares a field signal with its own delayed copy.

**What the main simulation demonstrates:** fully coherent two-beam interference, with perfect temporal coherence and deterministic spatial phase relationships assumed. Although the two arms contain delayed copies of one field, the simulation does not measure the source's statistical correlation versus optical path delay. Calling it a temporal self-coherence measurement would overstate what is demonstrated.

The EO example changes the correlation through imposed phase averaging; the Young example changes spatial coherence through its source model. Neither extracts the original laser's temporal self-coherence from a path-delay scan.
